## backend > beamline.py

In [24]:
from sympy import symbols, Matrix
import sympy as sp
import numpy as np
from scipy import interpolate
from scipy import optimize
import math
import torch

In [67]:
import os, sys
from pprint import pprint

# pprint(sys.path)

current_dir = os.path.abspath(os.getcwd())

project_root = os.path.abspath(os.path.join(current_dir, '../backend'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
pprint(sys.path)
from backend.beamline import *

['C:\\Users\\yi_lu\\my_files\\research\\FELsim\\backend',
 'C:\\Users\\yi_lu\\my_files\\research\\FELsim',
 'C:\\Users\\yi_lu\\programming_IDE\\PyCharm_2025.2.1\\plugins\\python-ce\\helpers\\pydev',
 'C:\\Users\\yi_lu\\programming_IDE\\PyCharm_2025.2.1\\plugins\\python-ce\\helpers\\jupyter_debug',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\python311.zip',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\DLLs',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\Lib',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim',
 '',
 'C:\\Users\\yi_lu\\AppData\\Roaming\\Python\\Python311\\site-packages',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\Lib\\site-packages']


## lattice > before init

In [26]:
# before
class lattice_test:
    E = 45  # Kinetic energy (MeV/c^2)
    ## this should be passed from ebeam
    E0 = 0.51099 # Electron rest energy (MeV/c^2)
    Q = 1.60217663e-19  # (C)
    M = 9.1093837e-31  # (kg)
    C = 299792458  # Celerity (m/s)
    f = 2856 * (10 ** 6)  # RF frequency (Hz)

    M_AMU = 1.66053906892E-27  # Atomic mass unit (kg)
    k_MeV = 1e-6 / Q  # Conversion factor (MeV / J)
    m_p = 1.67262192595e-27  # Proton Mass (kg)

    #  Dictionary of predefined particle properties [Mass (kg), Charge (C), Rest Energy (MeV)]
    PARTICLES = {"electron": [M, Q, (M * C ** 2) * k_MeV],
                      "proton": [m_p, Q, (m_p * C ** 2) * k_MeV]}

    # Relativistic gamma and beta factors, calculated based on current kinetic energy and rest energy
    gamma = (1 + (E / E0))
    beta = np.sqrt(1 - (1 / (gamma ** 2)))

    unitsF = 10 ** 6 # Units factor used for conversions from (keV) to (ns)
    color = 'none'  #Color of beamline element when graphed

In [27]:
# after
class lattice_test:
    E = torch.tensor(45)  # Kinetic energy (MeV/c^2)
    ## this should be passed from ebeam
    E0 = torch.tensor(0.51099) # Electron rest energy (MeV/c^2)
    Q = torch.tensor(1.60217663e-19)  # (C)
    M = torch.tensor(9.1093837e-31)  # (kg)
    C = torch.tensor(299792458)  # Celerity (m/s)
    f = torch.tensor(2856 * (10 ** 6))  # RF frequency (Hz)

    M_AMU = torch.tensor(1.66053906892E-27)  # Atomic mass unit (kg)
    k_MeV = 1e-6 / Q  # Conversion factor (MeV / J)
    m_p = torch.tensor(1.67262192595e-27)  # Proton Mass (kg)

    #  Dictionary of predefined particle properties [Mass (kg), Charge (C), Rest Energy (MeV)]
    PARTICLES = {"electron": [M, Q, (M * C ** 2) * k_MeV],
                      "proton": [m_p, Q, (m_p * C ** 2) * k_MeV]}

    # Relativistic gamma and beta factors, calculated based on current kinetic energy and rest energy
    gamma = (1 + (E / E0))
    beta = torch.sqrt(1 - (1 / (gamma ** 2)))

    unitsF = torch.tensor(10 ** 6) # Units factor used for conversions from (keV) to (ns)
    color = 'none'  #Color of beamline element when graphed

## lattice > setE()

In [28]:
# before
class lattice_test:
    ...
    def setE(self, E):
        '''
        Sets the kinetic energy (E) of the particle and updates dependent relativistic factors.

        Parameters
        ----------
        E : float
            New kinetic energy value (MeV/c^2).
        '''
        self.E = E
        self.gamma = (1 + (self.E/self.E0))
        self.beta = np.sqrt(1-(1/(self.gamma**2)))
    def setMQE(self, mass, charge, restE):
        '''
        Sets the mass, charge, and rest energy of the particle, and updates
        dependent relativistic factors.

        Parameters
        ----------
        mass : float
            The new mass of the particle in kg.
        charge : float
            The new charge of the particle in Coulombs.
        restE : float
            The new rest energy of the particle in MeV.
        '''
        self.M = mass
        self.Q = charge
        self.E0 = restE
        self.gamma = (1 + (self.E/self.E0))
        self.beta = np.sqrt(1-(1/(self.gamma**2)))

In [29]:
# after
class lattice_test:
    ...
    def setE(self, E):
        '''
        Sets the kinetic energy (E) of the particle and updates dependent relativistic factors.

        Parameters
        ----------
        E : float
            New kinetic energy value (MeV/c^2).
        '''
        self.E = E
        self.gamma = (1 + (self.E/self.E0))
        self.beta = torch.sqrt(1-(1/(self.gamma**2)))
        
    def setMQE(self, mass, charge, restE):
        '''
        Sets the mass, charge, and rest energy of the particle, and updates
        dependent relativistic factors.

        Parameters
        ----------
        mass : float
            The new mass of the particle in kg.
        charge : float
            The new charge of the particle in Coulombs.
        restE : float
            The new rest energy of the particle in MeV.
        '''
        self.M = mass
        self.Q = charge
        self.E0 = restE
        self.gamma = (1 + (self.E/self.E0))
        self.beta = torch.sqrt(1-(1/(self.gamma**2)))

## lattice > useMatrice()

In [30]:
# before
class lattice_test:
    ...

    def useMatrice(self, val, **kwargs):
        ''''
        Simulates the movement of given particles through its child segment by
        applying the segment's transfer matrix with numeric parameters.

        Parameters
        ----------
        val : np.ndarray
            A 2D NumPy array representing the particle states. Each row is a particle,
            and columns correspond to phase space coordinates (e.g., [x, x', y, y', z, z']).
        **kwargs : dict
            Other segment-specific numeric parameters (e.g., `length`, `current`)
            that might override the segment's default properties for this specific simulation.

        Returns
        -------
        list
            A 2D list where each inner list represents the transformed state of a particle
            after passing through the segment.

        '''
        np.random.seed(42)
        mat = np.random.randn(6, 6)
        npMat = np.array(mat).astype(np.float64)

        newMatrix = []
        for array in val:
            tempArray = np.matmul(npMat, array)
            newMatrix.append(tempArray.tolist())
        return newMatrix  #  return 2d list

In [31]:
l1 = lattice()

N = 1000
np.random.seed(42)
val = np.random.randn(N, 6)

matrix_1 = l1.useMatrice(val)
print(matrix_1)


TypeError: lattice.__init__() missing 1 required positional argument: 'length'

In [9]:
# after
class lattice_test:
    ...
    def useMatrice(self,val_tensor, **kwargs):
        '''
        [Refactored for PyTorch]
        Simulates the movement of given particles through its child segment by
        applying the segment's transfer matrix via batched tensor operations.
        '''

        # actual
        # mat = self.getSymbolicMatrice(numeric = True, **kwargs)
        # 1. get the matrix

        np.random.seed(42)
        mat = np.random.randn(6, 6)
        mat = torch.Tensor(mat)

        # transform into torch tensor
        mat_tensor = torch.tensor(np.array(mat).astype(np.float64),
                                  dtype=val_tensor.dtype,
                                  device=val_tensor.device)

        # parallelized matrix multiplication
        new_val_tensor = torch.matmul(val_tensor, mat_tensor.T)
        new_val_tensor = new_val_tensor.cpu().numpy().tolist()
        return new_val_tensor

In [10]:
torch.manual_seed(42)

l1 = lattice()
val_tensor = torch.from_numpy(val.copy())
matrix_2 = l1.useMatrice(val_tensor)
print(matrix_2)
# print(type(matrix_2[0]))

[[3.1146102048283435, 1.4181258438632196, -1.425284868117724, 0.6674368851259289, -0.2500874255957906, -2.081404457607018], [1.4181258059988235, 4.02930686539815, -0.25846825088297076, -2.696142048949793, 0.38382742892653493, 0.091146116965983], [-1.4252848803196976, -0.2584682744942937, 8.135274160451747, -0.4348653089904076, 1.94685452983672, -4.288244379764177], [0.6674369386882384, -2.6961420150951048, -0.43486536793358677, 7.052678142948627, -1.0590896983533569, -0.055651652177376824], [-0.25008747329319186, 0.3838274535131224, 1.946854584326459, -1.0590897052232806, 2.220443693433453, 0.013233683461308607], [-2.081404450142399, 0.09114606995027941, -4.288244245160411, -0.05565159910063284, 0.013233663382305883, 7.078959349599441], [-0.39876556260295865, -0.8657502443670494, 5.2861646263713284, 0.3926059907476306, 0.7780819238119346, -3.5476163070819204], [-2.2094294755330117, -0.3890063035934634, 4.301948786775742, -3.011447970468049, 1.4292045280382608, -1.376275551997347], [0.0

C:\Users\yi_lu\AppData\Local\Temp\ipykernel_45336\1567916368.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  mat_tensor = torch.tensor(np.array(mat).astype(np.float64),


In [18]:
print(len(matrix_1), len(matrix_2))
print(np.allclose(matrix_1, matrix_2, atol=1e-5))

1000 1000
True


## driftLattice(lattice) > getSymbolicMatrice
same for qpfLattice, qpdLattice, dipole, dipole_wedge, Beamline

In [12]:
# before
class driftLattice_test(lattice):
    ...
    def __init__(self, line = []):
        self.ORIGINFACTOR = 0.99

        a = {'first order decay': (self._frontModel, self._endModel)} # must work on later
        self.FRINGEDELTAZ = 0.01

        self.beamline = line
        self.totalLen = 0
        self.defineEndFrontPos()
    def getSymbolicMatrice(self, numeric=False, length=None):
        ...
        return mat

In [13]:
# after
class driftLattice_test(lattice):
    ...

    def getSymbolicMatrice(self, numeric=False, length=None):
        ...
        return torch.tensor(mat, dtype=torch.float32)
# same for qpfLattice, qpdLattice, dipole, dipole_wedge, Beamline

## Beamline > fringeField(lattice) > getSymbolicMatrice

In [ ]:
# before

class Beamline_Test:
    ...


    class fringeField(lattice):
        ...
        def getSymbolicMatrice(self, numeric = False, length = None, current = None):
            M56 = (l * self.f / (self.C * self.beta * self.gamma * (self.gamma + 1)))
            mat = Matrix([[1, l, 0, 0, 0, 0],
                        [0, 1, 0, 0, 0, 0],
                        [0, 0, 1, l, 0, 0],
                        [0, 0, 0, 1, 0, 0],
                        [0, 0, 0, 0, 1, M56],
                        [0, 0, 0, 0, 0, 1]])

            return torch.tensor(mat,dtype=torch.float32)


In [ ]:
# after

class Beamline_Test:
    ...

    class fringeField(lattice):
        ...
        def getSymbolicMatrice(self, numeric = False, length = None, current = None):
            M56 = (l * self.f / (self.C * self.beta * self.gamma * (self.gamma + 1)))
            mat = Matrix([[1, l, 0, 0, 0, 0],
                        [0, 1, 0, 0, 0, 0],
                        [0, 0, 1, l, 0, 0],
                        [0, 0, 0, 1, 0, 0],
                        [0, 0, 0, 0, 1, M56],
                        [0, 0, 0, 0, 0, 1]])

            return torch.tensor(mat,dtype=torch.float32)


## Beamline

In [ ]:
# before
class Beamline_Test:
    ...
    def __init__(self, line = []):
        self.ORIGINFACTOR = 0.99

        a = {'first order decay': (self._frontModel, self._endModel)} # must work on later
        self.FRINGEDELTAZ = 0.01

        self.beamline = line
        self.totalLen = 0
        self.defineEndFrontPos()

    #  Invariant, call at end of all functions
    def defineEndFrontPos(self):
        self.totalLen = 0
        for seg in self.beamline:
            seg.startPos = self.totalLen
            self.totalLen += seg.length
            seg.endPos = self.totalLen

    def interpolateData(self, xData, yData, interval):
        rbf = interpolate.Rbf(xData, yData)
        totalLen = xData[-1] - xData[0]
        xNew = np.linspace(xData[0], xData[-1], math.ceil(totalLen/interval) + 1)
        yNew = rbf(xNew)
        return xNew, yNew

    def _testModeOrder2end(self, x, origin, B0, a1, a2):
        return B0/(1+np.exp((a1*(x - origin)) + (a2*(x-origin)**2)))

    def _testModeOrder2front(self, x, origin, B0, a1, a2):
        return B0/(1+np.exp((a1*(-x - origin)) + (a2*(-x-origin)**2)))

    def _endModel(self, x, origin, B0, strength):
        return (B0/(1+np.exp((x-origin)*strength)))

    def _frontModel(self, x, origin, B0, strength):
        return (B0/(1+np.exp((-x+origin)*strength)))

    def reconfigureLine(self, interval = None):
        if interval is None:
            interval = self.FRINGEDELTAZ

        beamline = self.beamline
        totalLen = self.totalLen

        # zLine = np.linspace(0,totalLen,math.ceil(totalLen/interval)+1)
        # y_values = np.zeros_like(zLine)

        zLine = []
        i = 0
        while i <= totalLen:
            zLine.append(i)
            i += interval
        if not interval == (i - totalLen):
            zLine.append(totalLen)
        zLine = np.array(zLine)

        y_values = np.zeros_like(zLine)

        #  add to end of beam segments
        for segment in reversed(beamline):
            #  custom fringe field
            if isinstance(segment.fringeType, list):
                xData = segment.fringeType[0].copy()
                yData = segment.fringeType[1].copy()

                for i in range(len(xData)): xData[i] += segment.endPos  # adjust for z position

                params = self.endFit(xData,yData, segment.endPos)
                yfield = self._endModel(zLine, *params)

                # params = self.testendFit(xData, yData, segment.endPos)
                # yfield = self._testModeOrder2end(zLine, *params)

                zeroTracker = 0
                while (zLine[zeroTracker] < segment.endPos and zeroTracker < zLine.size):
                    yfield[zeroTracker] = 0
                    zeroTracker += 1

                y_values += yfield

            elif (segment.fringeType == 'first order decay'):
                B0 = 1 # temporary
                strength = 1 # temporary, must let user choose this
                yfield = self._endModel(zLine,
                                        segment.endPos-(np.log((1-self.ORIGINFACTOR)/self.ORIGINFACTOR)/strength),
                                        B0, strength)
                zeroTracker = 0
                while (zLine[zeroTracker] < segment.endPos and zeroTracker < zLine.size):
                    yfield[zeroTracker] = 0
                    zeroTracker += 1

                y_values += yfield


        #  add to the front of beam segments
        for segment in beamline:
            #  custom fringe field
            if isinstance(segment.fringeType, list):
                xData = segment.fringeType[0].copy()
                yData = segment.fringeType[1].copy()
                for i in range(len(xData)): xData[i] *= -1

                for i in range(len(xData)): xData[i] += segment.startPos
                params = self.frontFit(xData,yData, segment.startPos)
                yfield = self._frontModel(zLine, *params)

                # params = self.testFrontFit(xData, yData, segment.startPos)
                # yfield = self._testModeOrder2front(zLine, *params)

                zeroTracker = len(zLine)-1
                while (zLine[zeroTracker] > segment.startPos and zeroTracker >= 0):
                    yfield[zeroTracker] = 0
                    zeroTracker -= 1

                y_values += yfield

            elif (segment.fringeType == 'first order decay'):
                B0 = 1 # temporary?
                strength = 5 # temporary, must let user choose this
                yfield = self._frontModel(zLine,
                                        segment.startPos+(np.log((1-self.ORIGINFACTOR)/self.ORIGINFACTOR)/strength),
                                        B0, strength)

                zeroTracker = len(zLine)-1
                while (zLine[zeroTracker] > segment.startPos and zeroTracker >= 0):
                    yfield[zeroTracker] = 0
                    zeroTracker -= 1

                y_values += yfield

        #  Create each fringe field element, using right Riemann summnation
        i = 0
        while (i < len(beamline)):
            if isinstance(beamline[i], driftLattice):
                #  Find in x the value that first comes after start.pos
                index = np.searchsorted(zLine, beamline[i].startPos, side='right')

                totalDriftLen = beamline[i].length
                totalFringeLen = 0

                fringeLen = zLine[index] - beamline[i].startPos
                totalDriftLen -= fringeLen

                #  Convert drift segment into fringe field segments
                while (totalDriftLen >= 0 and index < len(y_values) - 1):
                    totalFringeLen += fringeLen
                    fringe = self.fringeField(fringeLen, y_values[index])
                    beamline.insert(i, fringe)

                    i += 1 # for adding a fringe field
                    index += 1 # Get next y values
                    fringeLen = zLine[index] - zLine[index-1]
                    totalDriftLen -= fringeLen

                beamline[i].length -= totalFringeLen

                #  Convert last portion of drift portion to fringe field and remove
                if (beamline[i].length > 0 and index < len(y_values)):
                    fringe = self.fringeField(beamline[i].length, y_values[index])
                    beamline.insert(i, fringe)
                    i += 1
                    #WIP adding to final leftover interval

                beamline.pop(i)
            i += 1

        self.defineEndFrontPos()
        return zLine, y_values


In [62]:
# after
class Beamline_Test:
    class fringeField(lattice):
        B = 0 #  Teslas
        color = 'brown'

        def __init__(self, length, fieldStrength, current = 0):
            super().__init__(length)
            self.B = fieldStrength
            # self.current = current

        #temporarily use drift matrice for testing
        def getSymbolicMatrice(self, numeric = False, length = None, current = None):
            l = None
            I = None
            if length is None:
                l = self.length
            else:
                if numeric: l = length  # length should be number
                else: l = symbols(length, real = True)  # length should be string

            # if current is None:
            #     I = self.current
            # else:
            #     if numeric: I = current  # current should be number
            #     else: I = symbols(current, real = True)  # current should be string

            # self.k = sp.Abs((self.Q*self.G*I * self.fringeDecay )/(self.M*self.C*self.beta*self.gamma))
            # self.theta = sp.sqrt(self.k)*l

            # M11 = sp.cos(self.theta)
            # M21 = (-(sp.sqrt(self.k)))*sp.sin(self.theta)
            # M22 = sp.cos(self.theta)
            # M33 = sp.cosh(self.theta)
            # M43 = sp.sqrt(self.k)*sp.sinh(self.theta)
            # M44 = sp.cosh(self.theta)
            # M56 = -(l * self.f / (self.C * self.beta * self.gamma * (self.gamma + 1)))

            # if I == 0:
            #     M12 = l
            #     M34 = l
            # else:
            #     M34 = sp.sinh(self.theta)*(1/sp.sqrt(self.k))
            #     M12 = sp.sin(self.theta)*(1/sp.sqrt(self.k))

            # mat =  Matrix([[M11, M12, 0, 0, 0, 0],
            #                 [M21, M22, 0, 0, 0, 0],
            #                 [0, 0, M33, M34, 0, 0],
            #                 [0, 0, M43, M44, 0, 0],
            #                 [0, 0, 0, 0, 1, M56],
            #                 [0, 0, 0, 0, 0, 1]])

            # return mat

            M56 = (l * self.f / (self.C * self.beta * self.gamma * (self.gamma + 1)))
            mat = Matrix([[1, l, 0, 0, 0, 0],
                        [0, 1, 0, 0, 0, 0],
                        [0, 0, 1, l, 0, 0],
                        [0, 0, 0, 1, 0, 0],
                        [0, 0, 0, 0, 1, M56],
                        [0, 0, 0, 0, 0, 1]])

            return torch.tensor(mat,dtype=torch.float32)


        def __str__(self) -> str:
            return f"Fringe field segment {self.length} m long with a magnetic field of {self.B} teslas"

    def csvToBeamline(self, csv):
        with open(csv, newline='') as file:
            reader = csv.reader(file)

    def __init__(self, line = []):
        self.ORIGINFACTOR = torch.tensor(0.99)

        a = {'first order decay': (self._frontModel, self._endModel)} # must work on later
        self.FRINGEDELTAZ = torch.tensor(0.01)

        self.beamline = line
        self.totalLen = torch.zeros(1)
        self.defineEndFrontPos()

    def defineEndFrontPos(self):
        """
        Vectorized calculation of beamline positions using cumulative sums.
        Replaces the Python 'for' loop with a single PyTorch pass.
        """
        lengths = torch.tensor([seg.length for seg in self.beamline])


        end_positions = torch.cumsum(lengths, dim=0)


        start_positions = torch.cat([torch.tensor([0.0]), end_positions[:-1]])


        for i, seg in enumerate(self.beamline):
            seg.startPos = start_positions[i].item()
            seg.endPos = end_positions[i].item()


        self.totalLen = end_positions[-1].item() if len(end_positions) > 0 else 0

    def interpolateData(self, xData, yData, interval):
        rbf = interpolate.Rbf(xData, yData)
        totalLen = xData[-1] - xData[0]
        xNew = torch.linspace(xData[0], xData[-1], math.ceil(totalLen/interval) + 1)
        yNew = rbf(xNew)
        return xNew, yNew

    def _testModeOrder2end(self, x, origin, B0, a1, a2):
        return B0/(1+torch.exp((a1*(x - origin)) + (a2*(x-origin)**2)))

    # ❌not changed, just for code to run
    def testFrontFit(self, xData, yData, pos):
        endParams, _ = optimize.curve_fit(self._testModeOrder2front, xData, yData, p0= [pos, 1, 1, 1], maxfev=50000)
        return endParams

    # ❌not changed, just for code to run
    def testendFit(self, xData, yData, pos):
        endParams, _ = optimize.curve_fit(self._testModeOrder2end, xData, yData, p0= [pos, 1, 1, 1], maxfev=50000)
        print(endParams)
        return endParams


    def _testModeOrder2front(self, x, origin, B0, a1, a2):
        return B0/(1+torch.exp((a1*(-x - origin)) + (a2*(-x-origin)**2)))

    def _endModel(self, x, origin, B0, strength):
        return (B0/(1+torch.exp((x-origin)*strength)))


    def _frontModel(self, x, origin, B0, strength):
        return (B0/(1+torch.exp((-x+origin)*strength)))

    # ❌not changed, just for code to run
    def frontFit(self, xData, yData, pos):
        endParams, _ = optimize.curve_fit(self._frontModel, xData, yData, p0= [pos, 1, 1], maxfev=50000)
        return endParams

    # ❌not changed, just for code to run
    def endFit(self, xData, yData, pos):
        endParams, _ = optimize.curve_fit(self._endModel, xData, yData, p0= [pos, 1, 1], maxfev=50000)
        return endParams

    def reconfigureLine(self, interval=None):
        """
        Reconfigures the beamline by discretizing drift spaces into fringe field segments.
        Uses PyTorch vectorized operations to calculate total magnetic field distribution.
        All logic is computed in tensors to avoid expensive Python loops.
        """
        if interval is None:
            interval = self.FRINGEDELTAZ

        # 1. Vectorized zLine generation using torch.linspace
        # Replacing: while i <= totalLen: zLine.append(i)
        num_points = int(torch.ceil(self.totalLen / interval)) + 1
        z_line = torch.linspace(0, self.totalLen, num_points, dtype=torch.float32)
        y_values = torch.zeros_like(z_line)

        # 2. Vectorized Field Superposition (End Model)
        # Replaces 'zeroTracker' while-loops with Boolean Masking
        for segment in self.beamline:
            if segment.fringeType == 'first order decay':
                b0 = 1.0 # Temporary constant
                strength = 1.0
                # Calculate the origin point for the decay model
                origin = segment.endPos - (torch.log((1 - self.ORIGINFACTOR) / self.ORIGINFACTOR) / strength)

                # Compute the entire field for all z points simultaneously
                y_field = b0 / (1 + torch.exp((z_line - origin) * strength))

                # Apply Mask: zero out fields before the segment end (Parallelized)
                y_field[z_line < segment.endPos] = 0
                y_values += y_field

        # 3. Vectorized Field Superposition (Front Model)
        for segment in self.beamline:
            if segment.fringeType == 'first order decay':
                b0 = 1.0
                strength = 5.0
                origin = segment.startPos + (torch.log((1 - self.ORIGINFACTOR) / self.ORIGINFACTOR) / strength)

                y_field = b0 / (1 + torch.exp((-z_line + origin) * strength))

                # Apply Mask: zero out fields after the segment start
                y_field[z_line > segment.startPos] = 0
                y_values += y_field

        # 4. Optimized Reconstruction (Building a new list instead of in-place insert)
        # Python list.insert() is O(N), repeatedly inserting makes it O(N^2)
        new_beamline = []
        for seg in self.beamline:
            if not isinstance(seg, driftLattice):
                new_beamline.append(seg)
            else:
                # Vectorized lookup of the drift's range within the z_line
                mask = (z_line >= seg.startPos) & (z_line <= seg.endPos)
                z_slice = z_line[mask]
                y_slice = y_values[mask]

                # Sub-divide the drift into fringeField micro-segments
                for j in range(1, len(z_slice)):
                    l_slice = z_slice[j] - z_slice[j-1]
                    # Creating segment objects
                    new_beamline.append(self.fringeField(l_slice.item(), y_slice[j].item()))

                # Handle floating point residual for the last segment of the drift
                remaining_l = seg.length - (z_slice[-1] - z_slice[0])
                if remaining_l > 1e-12:
                    new_beamline.append(self.fringeField(remaining_l.item(), y_slice[-1].item()))

        self.beamline = new_beamline
        self.defineEndFrontPos() # Refresh position metadata

        return z_line.numpy(), y_values.numpy()

In [63]:
# test the change of defineEndFrontPos() works

test_line = [
    driftLattice(length=0.5),
    qpfLattice(current=5.0, length=0.1),
    driftLattice(length=1.0)
]


my_beamline = Beamline(line=test_line)
my_beamline2 = Beamline_Test(line=test_line)

# print(f"--- beam line position evaluation ---")
# for i, seg in enumerate(my_beamline.beamline):
#     print(f"segment {i} ({type(seg).__name__}):")
#     print(f"  length: {seg.length} m")
#     print(f"  start: {seg.startPos:.3f} m")
#     print(f"  end: {seg.endPos:.3f} m")
#     print("-" * 25)
#
# print(f"total length: {my_beamline.totalLen:.3f} m")

for (i,seg1),(j,seg2) in zip(enumerate(my_beamline.beamline), enumerate(my_beamline2.beamline)):
    assert type(seg1) == type(seg2), f"Segment {i} type mismatch: {type(seg1).__name__} vs {type(seg2).__name__}"
    assert seg1.length == seg2.length, f"Segment {i} length mismatch: {seg1.length} vs {seg2.length}"
    assert seg1.startPos == seg2.startPos, f"Segment {i} start position mismatch: {seg1.startPos} vs {seg2.startPos}"
    assert seg1.endPos == seg2.endPos, f"Segment {i} end position mismatch: {seg1.endPos} vs {seg2.endPos}"


In [64]:
# test interpolateData
x_raw = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
y_raw = np.sin(np.pi * x_raw)

test_interval = 0.01

x_new1, y_new1 = my_beamline.interpolateData(x_raw, y_raw, test_interval)
x_new2, y_new2 = my_beamline2.interpolateData(x_raw, y_raw, test_interval)

print(np.allclose(x_new1, x_new2))
print(np.allclose(y_new1, y_new2))

True
True


In [66]:
# test reconfigureline
z1, y1=my_beamline.reconfigureLine()
z2, y2=my_beamline2.reconfigureLine()
print(np.allclose(z1, z2))
print(np.allclose(y1, y2))

True
True
